## Gradio Day

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
import gradio as gr

In [ ]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging
# You can choose whichever providers you like - or all Ollama

load_dotenv(override=True)
groq_api_key = os.getenv("GROQ_API_KEY")
google_api_key = os.getenv("GEMINI_API_KEY")
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openai_api_key = os.getenv('OPENAI_API_KEY')

if groq_api_key:
  print(f"GroqAI API Key Exists and begin with {groq_api_key[:2]}")
else:
  print("Groq API Key doesn't exist")

if openai_api_key:
  print(f"OpenAI API Key ecists and begins {openai_api_key[:8]}")
else:
  print("OpenAI API Key does not exist")

if google_api_key:
  print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
  print("Google API Key does not exist")

if openrouter_api_key:
  print(f"OpenRouter API Key exists and begins {openrouter_api_key[:8]}")
else:
  print("OpenRouter API Key does not exist")

In [ ]:
#--- For Gemini Model---
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
for model in gemini.models.list():
  print(model.id)

In [ ]:
# Connect to OpenAI client library
openai = OpenAI()
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"

openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(base_url=ollama_url, api_key="ollama")

In [ ]:
#let's wrap a call to Gemini
system_message = "You are a helpful assistant"

def message_gemini(prompt):
  messages = [
    {
      "role":"system", "content": system_message
    },
    {
      "role": "user", "content": prompt
    }
  ]

  response = gemini.chat.completions.create(model = "gemini-3.1-flash-lite", reasoning_effort="none", messages = messages)
  return response.choices[0].message.content

In [ ]:
# This can reveal the "training cut off", or the most recent data in the training data
message_gemini("what is todays date?")

## User Interface Time!

In [ ]:
# Here's the simple function
def shout(text):
  print(f"Shout has been called with input {text}")
  return text.upper()

In [ ]:
shout("hello")

In [ ]:
gr.Interface(fn=shout, inputs='textbox', outputs='textbox', flagging_mode="manual").launch()

## Note: Using Gradio's Share tool
it's a really cool way to share your Gradio UI with others. This deploys your gradio app as a demo on gradio's website, and then allows gradio to call the 'shout' function. This users an advanced technology known as "HTTP tunneling" (like ngrok for people who know it) which isn't allowed by many Antivirus Programs and corporate environments. If you get an error, just skip the next cell.

In [ ]:
# Adding share=True means that it can be accessed publically
# A more permanent hostiing is available using a platform called Spaces from HuggingFace, which we will touch on next week.
# NOTE: Some Anti-Virus software & Corporate Firewalls might not like you using share=True.
# If you're at work on a work network, I suggest skip this test.

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(share=True)

In [ ]:
# Adding inbrowser=True opens up a new browser window automatically
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True)

## Adding Authentication
Gradio makes it very easy to have usreids and passwords
Obviously if you use this, have it look properly in a secure place for passwords! At a minimum, use your .env

In [ ]:
gr.Interface(fn=shout, inputs='textbox', outputs='textbox', flagging_mode='never').launch(inbrowser=True, auth=('cytor', 'mil123'))

## Forcing Dark Mode
Gradio appears in light mode or dark mode depending on the settings og the browser and computer. THere is a way to force gradio to appear in dark mode, but Gradio recommends against this as it should be a user preference (particularly for accessibility reason). But if you wish  to force dark mode for your screens. this is how it has to be done.

In [ ]:
# Define this variable and then pass js=force_dark_mode when creating the Interface.
force_dark_mode = """
function refresh() {
  const url = new URL(window.location);
  if (url.searchParams.get('__theme') !== 'dark') {
      url.searchParams.set('__theme', 'dark');
      window.location.href = url.href;
  }
}
"""
gr.Interface(fn=shout, inputs='textbox', outputs="textbox", flagging_mode="never", js=force_dark_mode).launch(inbrowser=False)

In [ ]:
# Adding a little bit more:

message_input = gr.Textbox(label="Your message:", info="Enter a message to be shouted", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
  fn= shout,
  title='Shout',
  inputs=[message_input],
  outputs=[message_output],
  examples=['hello', 'howdy'],
  flagging_mode='never'
)
view.launch()

In [ ]:
# And now - Changing the function from "shout" to "message_gemini"

message_input = gr.Textbox(label="Your Message", info = "Enter a message for gemini", lines = 7)
message_output = gr.Textbox(label="Response", lines=8)

view = gr.Interface(
  fn=message_gemini,
  title="Gemini",
  inputs=[message_input],
  outputs=[message_output],
  examples=['hello', 'howdy'],
  flagging_mode='never'
)
view.launch()

In [ ]:
# Let's use Markdown
# Are you wondering why it makes difference to set system_message when it's not referred to in the code below it?
# I'm taking advantage of system_message being a global cvariable, used back in the message_gemini function (go take a look)
# Not a great software engineering practice, but quite common during Jupyter Lab R&D!

system_message = "You are a Helpful assistant that responds in markdown without code blocks"

message_input = gr.Textbox(label='Your Message:', info="enter a message for gemini", lines =7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
  fn=message_gemini,
  title="Gemini",
  inputs=[message_input],
  outputs=[message_output],
  examples=[
    "Explain the Transformer Architecture to a layperson",
    "Explain the Transformer Architecture to an aspiring AI engineer",
  ],
  flagging_mode="never"
)
view.launch()

In [ ]:
# Let's create a call that streams back results
# If you'd like a refresher on Generators (the 'yield' keyword),
# look at some Intermediate Python guide from W3School or your preferred website

def stream_gemini(prompt):
  messages = [
    {
      "role": "system",
      "content": system_message
    },
    {
      "role": "user",
      "content": prompt
    }
  ]

  stream = gemini.chat.completions.create(
    model='gemini-3.1-flash-lite',
    reasoning_effort='none',
    messages=messages,
    stream=True
  )

  result = ""
  for chunk in stream:
    content = chunk.choices[0].delta.content
    if content:
      result += content
      yield result

In [ ]:
message_input = gr.Textbox(label="Your Message", info="Enter a message for gemini", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
  fn=stream_gemini,
  title="Gemini",
  inputs=[message_input],
  outputs=[message_output],
  examples=[
    "Explain the Transformer architecture to a layperson",
    "Explain the Transformer Architecture to an aspiring AI engineer",
  ],
flagging_mode='never'
)
view.launch()

In [ ]:
# # For Claude If you are using claude

# def stream_claude(prompt):
#     messages = [
#         {"role": "system", "content": system_message},
#         {"role": "user", "content": prompt}
#       ]
#     stream = anthropic.chat.completions.create(
#         model='claude-sonnet-5-5',
#         messages=messages,
#         stream=True
#     )
#     result = ""
#     for chunk in stream:
#         result += chunk.choices[0].delta.content or ""
#         yield result

# message_input = gr.Textbox(label="Your message:", info="Enter a message for Claude Sonnet 5.5", lines=7)
# message_output = gr.Markdown(label="Response:")

# view = gr.Interface(
#     fn=stream_claude,
#     title="Claude", 
#     inputs=[message_input], 
#     outputs=[message_output], 
#     examples=[
#         "Explain the Transformer architecture to a layperson",
#         "Explain the Transformer architecture to an aspiring AI engineer",
#         ], 
#     flagging_mode="never"
#     )
# view.launch()

In [ ]:
# For Groq

groq = OpenAI(api_key=groq_api_key, base_url=groq_url)

def stream_groq(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = groq.chat.completions.create(
        model='openai/gpt-oss-120b',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

## And Now Getting Fancy
Remeber  to check the Intermediate Python from any Website (ex: W3School) if you are unsuer about generators and yield.

In [ ]:
def stream_model(prompt, model):
  if model == "Gemini":
    result = stream_gemini(prompt)
  elif model == 'Groq':
    result = stream_groq(prompt)
  else:
    raise ValueError("Unknown model")
  yield from result

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["Gemini", "Groq"], label="Select model", value="Gemini")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="LLMs", 
    inputs=[message_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Explain the Transformer architecture to a layperson", "Gemini"],
            ["Explain the Transformer architecture to an aspiring AI engineer", "Groq"]
        ], 
    flagging_mode="never"
    )
view.launch()

## Building a Company brochure Generator

In [ ]:
from scraper import fetch_website_contents

In [ ]:
# Again this is typical Experimental mindset - I'm changing the global variable we used above:

system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [ ]:
def stream_brochure(company_name, url, model):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    if model=="Gemini":
        result = stream_gemini(prompt)
    elif model=="Groq":
        result = stream_groq(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [ ]:
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["Gemini", "Groq"], label="Select model", value="Gemini")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator", 
    inputs=[name_input, url_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Hugging Face", "https://huggingface.co", "Gemini"],
            ["Edward Donner", "https://edwarddonner.com", "Groq"]
        ], 
    flagging_mode="never"
    )
view.launch()

# Gradio Resources

If you'd like to go deeper on Gradio, check out the amazing documentation = a wonderful rabbit hole.

"https://www.gradio.app/guides/quickstart"
